# Somo la 13 - Kumbukumbu ya Wakala


## Mpangilio

Daftari hili linaonyesha jinsi ya kujenga wakala wa uhifadhi wa usafiri wenye **kumbukumbu ya kudumu** ukitumia **Mfumo wa Wakala wa Microsoft** (MAF).

Utajifunza jinsi aina tofauti za kumbukumbu za wakala — kazi, muda mfupi, na muda mrefu — zinavyoathiri jinsi wakala anavyohifadhi na kutumia taarifa katika mazungumzo.

**Mahitaji ya awali:**
- Mradi wa Azure AI Foundry wenye mfano wa mazungumzo uliotumika (mfano `gpt-4o-mini`).
- Umeingia kwenye Azure CLI — endesha `az login` kwenye terminal yako.
- `AZURE_AI_PROJECT_ENDPOINT` — kiunganishi cha mradi wako wa Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jina la mfano uliotumika.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Aina za Kumbukumbu za Wakala

Wakali wa AI wanaweza kutumia aina tofauti za kumbukumbu, kila moja ikiwa na madhumuni yake maalum:

### Kumbukumbu ya Kazi
Msingi wa mazungumzo yenyewe — ujumbe unaobadilishanwa katika kikao kimoja. Wakali wanaweza kurejelea nyuma ujumbe wa awali katika msingi huo huo ili kudumisha muendelezo mzuri. Katika MAF hii huundwa kupitia **`agent.create_session()`**, ambayo hurejesha `AgentSession`.

### Kumbukumbu ya Muda Mfupi
Taarifa ambazo hudumu kwa muda wa kazi au kikao lakini hazihifadhiwi kwa kudumu. Kwa mfano, wakala anaweza kukusanya ukweli wakati wa mazungumzo ya upangaji wa mizunguko mingi na kuyatumia kutoa ratiba ya mwisho.

### Kumbukumbu ya Muda Mrefu
Mapendeleo na ukweli ambao hudumu **katika vikao mbalimbali**. Mtumiaji anayerudi hafai kurudia vikwazo vya mlo au mtindo wa safari. Kumbukumbu ya muda mrefu kawaida huungwa mkono na hifadhi ya nje — hifadhidata, faili, au faharasa ya vector — na kuletwa kwa wakala kupitia zana.


## Kumbukumbu ya Kazi na Vikao

Aina rahisi kabisa ya kumbukumbu ni kikao cha mazungumzo. Unapopita kikao hicho hicho (kilichoundwa kupitia `agent.create_session()`) kwa simu za mfululizo za `agent.run()`, wakala anaona historia yote ya mazungumzo hayo na anaweza kukumbuka maelezo ya awali.

Tufanye wakala wa usafiri na kuonyesha kumbukumbu ya kazi.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Wakala alikumbuka bajeti kwa usahihi kwa sababu ujumbe wote wawili unashiriki kikao hicho hicho. Hii ni **kumbukumbu ya kazi** — ipo tu kwa muda wa kikao hicho.

### Nini hutokea na muktadha mpya?

Ikiwa tutaunda kikao **kipya**, wakala hana kumbukumbu ya mazungumzo ya awali:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Mfano wa Kumbukumbu ya Muda Mrefu

Ili kukumbuka mapendeleo ya mtumiaji **katika vikao tofauti**, tunahitaji hifadhi ya kudumu inayopatikana nje ya muktadha wa mazungumzo. Wakala hufikia hifadhi hii kupitia **zana** — kazi anaweza kuita kuhifadhi na kupata taarifa.

Chini yetu tunatekeleza hifadhi rahisi ya mapendeleo ya ndani ya kumbukumbu (katika utengenezaji ungeziwekea nyuma na hifadhidata au fahirisi ya vekta) na kuizua kama zana ambazo wakala anaweza kutumia.

### Muundo
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Hali ya 1 — Mtumiaji wa mara ya kwanza anaweka kitabu cha safari ya kumbukumbu

Sarah anatembelea kwa mara ya kwanza. Mwakala anapaswa kuhifadhi vipaumbele vyake kupitia zana na kutumia hivyo kupendekeza hoteli.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Tukio la 2 — Sarah anarudi baada ya wiki kadhaa

Sarah anaanzisha **mdomo mpya kabisa** (akigundua kikao kipya). Kumbukumbu ya kazi ni tupu, lakini hifadhi ya mapendeleo ya muda mrefu bado ina taarifa zake. Wakala anapaswa kuitafuta na kuitumia kubinafsisha mapendekezo.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Muhtasari

Katika somo hili ulijifunza aina tatu za kumbukumbu za wakala na jinsi ya kuzitekeleza kwa kutumia Microsoft Agent Framework:

| Aina ya Kumbukumbu | Utaratibu wa MAF | Muda wa Kuishi |
|---|---|---|
| **Inayofanya kazi** | `agent.create_session()` | Mazungumzo moja |
| **Muda mfupi** | Muktadha uliokusanywa ndani ya thread | Kazi / kikao kimoja |
| **Muda mrefu** | Hifadhi ya nje inayopatikana kupitia shughuli za `@tool` | Kati ya vikao |

### Muhimu Kumbuka
1. **`agent.create_session()`** hutoa kumbukumbu inayofanya kazi — wakala anaona historia kamili ya mazungumzo ndani ya kikao.
2. **Vikao vipya hupoteza muktadha** — bila kumbukumbu ya muda mrefu wakala hawezi kukumbuka mazungumzo ya zamani.
3. **Shughuli za `@tool`** huunganisha pengo — huruhusu wakala kuhifadhi na kurejesha taarifa kutoka kwenye hifadhi endelevu.
4. **Uthabiti wa kibinafsi huboreka kwa muda** — upendeleo zaidi zikihifadhiwa, mapendekezo ya wakala yanakuwa bora zaidi.

### Matumizi halisi ya Dunia
- **Huduma kwa Wateja**: Kumbuka historia ya mteja na mapendeleo
- **Msaidizi wa Kibinafsi**: Hifadhi muktadha kati ya siku au wiki
- **Huduma za Afya**: Fuatilia taarifa na mapendeleo ya mgonjwa
- **Biashara mtandaoni**: Ununuzi uliobinafsishwa kulingana na historia

### Hatua Zifuatazo
- Badilisha kamusi ya kumbukumbu ya ndani na hifadhidata au hifadhi ya vekta (mfano: Azure AI Search)
- Ongeza uisha wa kumbukumbu kwa taarifa nyeti kwa muda
- Jenga mifumo ya wakala wengi yenye kumbukumbu zilizoshirikiwa
- Chunguza daftari la Cognee kwa kumbukumbu zinazojengwa kwa grafu ya maarifa


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kiarifa cha Kubadili Lugha**:  
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kufanikisha usahihi, tafadhali fahamu kuwa tafsiri za moja kwa moja zinaweza kuwa na makosa au kutokamilika. Hati ya asili katika lugha yake ya asili inapaswa kuzingatiwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu na ya mwanadamu inashauriwa. Hatutojishughulisha na makosa au ufafanuzi usio sahihi unaotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
